<a href="https://colab.research.google.com/github/nanpolend/machine-learning/blob/master/notebookaa7aGPT5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

arc_prize_2025_path = kagglehub.competition_download('arc-prize-2025')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ARC Prize 2025（ARC‑AGI‑2）
可提交的 Python 基線解題器（單檔版本，便於 Kaggle Notebook 直接貼上使用）

設計目標（比賽 + 公開審閱友善）
- 開放授權：本檔案以 MIT-0 授權釋出（公共領域等效）。
- 可重現性：固定亂數種子、嚴格資源上限（時間、擴展寬度、深度）。
- 可審閱性：具體註解、明確函式責任、最少外部依賴（只用 Python 標準庫 + numpy）。
- 可提交性：產出 Kaggle 要求格式的 submission.json；支援 pass@2 兩次嘗試。
- 可擴充性：以「微型 DSL + 條件化管線 + Beam Search」為核心，易於增添運算元與啟發式。

注意：此為強化版「規則/程式合成」起步基線，著重乾淨結構與審閱友好，而非極致 SOTA 成績。
在 L4x4（約 96GB）與 12 小時限制下，請自行調整 BEAM、DEPTH、TIMEOUT 等參數以平衡效率/準確率。

使用方式（Kaggle Notebook）
1) 在「/kaggle/input/arc-prize-2025」存在官方資料集時，直接執行本檔：
   python arc_baseline.py --data-root /kaggle/input/arc-prize-2025 --out submission.json
2) 或在 Notebook 內：
   from arc_baseline import main; main(["--data-root","/kaggle/input/arc-prize-2025"])

輸出：生成 submission.json（字典：task_id -> [ans1, ans2]），每個答案為 2D 陣列（list[list[int]]）。
"""

from __future__ import annotations
import argparse
import json
import math
import os
import random
import sys
import time
from dataclasses import dataclass
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

# ---------------------------
# 全域參數（可按需調整）
# ---------------------------
SEED = 1337
RNG = random.Random(SEED)
np.random.seed(SEED)

# 搜索限制（比賽效率優先）
MAX_DEPTH = 4          # 最大程式長度（可視時間調 3~6）
BEAM_SIZE = 64         # Beam 寬度（可視 GPU/時間調整）
CANDIDATES_PER_STEP = 256  # 每步生成多少候選（越大越準，越慢）
TIMEOUT_PER_TASK = 20.0    # 每個 task 的硬時限（秒）

# pass@2：保留前 2 名方案
NUM_ATTEMPTS = 2

# ---------------------------
# 資料讀取與序列化工具
# ---------------------------

def grid_to_list(grid: np.ndarray) -> List[List[int]]:
    return grid.astype(int).tolist()


def list_to_grid(x: List[List[int]]) -> np.ndarray:
    return np.array(x, dtype=np.int64)


def load_arc_split(data_root: str, split_name: str) -> Dict[str, Dict[str, Any]]:
    """讀取 ARC‑AGI‑2 的指定 split（public/evaluation 半私有等以官方釋出為準）。
    預期結構：data_root/{split_name}/{task_id}.json
    每個 JSON 包含：{"train": [...], "test": [...]}，皆為圖格資料。
    """
    split_dir = os.path.join(data_root, split_name)
    tasks: Dict[str, Dict[str, Any]] = {}
    if not os.path.isdir(split_dir):
        raise FileNotFoundError(f"找不到分割：{split_dir}")
    for fn in sorted(os.listdir(split_dir)):
        if not fn.endswith('.json'):
            continue
        task_id = fn.replace('.json', '')
        with open(os.path.join(split_dir, fn), 'r') as f:
            tasks[task_id] = json.load(f)
    return tasks


# ---------------------------
# 微型 DSL：格狀圖形運算原語
# ---------------------------
# 設計理念：
#   - 原語保持簡潔可組合；
#   - 儘量「不需參數或少參數」，利於暴力/啟發式搜尋；
#   - 涵蓋旋轉/翻轉/轉置、物件抽取、重著色、填充、對齊與縮放等常見規則；

Color = int  # 0..9
Grid = np.ndarray  # (H,W) int64


def rot90(g: Grid) -> Grid:
    return np.rot90(g, k=1)


def rot180(g: Grid) -> Grid:
    return np.rot90(g, k=2)


def rot270(g: Grid) -> Grid:
    return np.rot90(g, k=3)


def flip_h(g: Grid) -> Grid:
    return np.fliplr(g)


def flip_v(g: Grid) -> Grid:
    return np.flipud(g)


def transpose(g: Grid) -> Grid:
    return g.T.copy()


def identity(g: Grid) -> Grid:
    return g.copy()


def majority_color(g: Grid) -> Color:
    vals, cnts = np.unique(g, return_counts=True)
    return int(vals[np.argmax(cnts)])


def fill_majority(g: Grid) -> Grid:
    c = majority_color(g)
    return np.full_like(g, c)


def bbox_of_color(g: Grid, c: Color) -> Optional[Tuple[int, int, int, int]]:
    ys, xs = np.where(g == c)
    if len(ys) == 0:
        return None
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    return y0, y1 + 1, x0, x1 + 1  # 半開區間


def crop_bbox(g: Grid, box: Tuple[int, int, int, int]) -> Grid:
    y0, y1, x0, x1 = box
    return g[y0:y1, x0:x1].copy()


def paste_center(canvas: Grid, patch: Grid) -> Grid:
    H, W = canvas.shape
    h, w = patch.shape
    out = canvas.copy()
    y0 = (H - h) // 2
    x0 = (W - w) // 2
    out[y0:y0+h, x0:x0+w] = patch
    return out


def object_mask(g: Grid, c: Color) -> Grid:
    return (g == c).astype(np.int64)


def shrink_to_bbox(g: Grid) -> Grid:
    # 去除外圍背景（以多數色當背景）
    bg = majority_color(g)
    box = bbox_of_color(g, bg)
    if box is None:
        return g.copy()
    # bbox_of_color 對 bg 會抓到整個畫面，需反向處理
    # 我們改抓非背景的最小包圍盒
    ys, xs = np.where(g != bg)
    if len(ys) == 0:
        return g.copy()
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    return g[y0:y1, x0:x1].copy()


def tile_repeat(patch: Grid, H: int, W: int) -> Grid:
    ph, pw = patch.shape
    ry = math.ceil(H / ph)
    rx = math.ceil(W / pw)
    tiled = np.tile(patch, (ry, rx))[:H, :W]
    return tiled.copy()

# 混色/重著色：以頻率對映

def recolor_by_rank(g: Grid) -> Grid:
    vals, cnts = np.unique(g, return_counts=True)
    order = vals[np.argsort(-cnts)]  # 由多到少
    mapping = {int(c): int(i % 10) for i, c in enumerate(order)}
    out = g.copy()
    for c_old, c_new in mapping.items():
        out[g == c_old] = c_new
    return out

# 物件抽取與拼接（簡化版）：取出最常見顏色物件後，對其做規則

def extract_dominant_object(g: Grid) -> Grid:
    c = majority_color(g)
    return crop_bbox(g, bbox_of_color(g, c)) if bbox_of_color(g, c) else g.copy()

# ---------------------------
# 程式表示與評分
# ---------------------------
Primitive = Callable[[Grid], Grid]

PRIMITIVES: List[Tuple[str, Primitive]] = [
    ("identity", identity),
    ("rot90", rot90),
    ("rot180", rot180),
    ("rot270", rot270),
    ("flip_h", flip_h),
    ("flip_v", flip_v),
    ("transpose", transpose),
    ("fill_majority", fill_majority),
    ("shrink_to_bbox", shrink_to_bbox),
    ("recolor_by_rank", recolor_by_rank),
]

# 提供幾個二階操作（先抽物件再放置）作為 compound 原語

def op_dominant_center(g: Grid) -> Grid:
    patch = extract_dominant_object(g)
    canvas = np.full_like(g, majority_color(g))
    return paste_center(canvas, patch)


def op_tile_dominant(g: Grid) -> Grid:
    patch = extract_dominant_object(g)
    return tile_repeat(patch, *g.shape)

PRIMITIVES += [
    ("dominant_center", op_dominant_center),
    ("tile_dominant", op_tile_dominant),
]


@dataclass(frozen=True)
class Program:
    ops: Tuple[str, ...]

    def run(self, g: Grid) -> Grid:
        out = g
        for name in self.ops:
            # 動態查找 primitive
            fn = dict(PRIMITIVES)[name]
            out = fn(out)
        return out

    def describe(self) -> str:
        return " ∘ ".join(self.ops) if self.ops else "(identity)"


def program_candidates(prev: Program) -> Iterable[Program]:
    for (name, _fn) in PRIMITIVES:
        yield Program(prev.ops + (name,))


def grid_equal(a: Grid, b: Grid) -> bool:
    return a.shape == b.shape and np.array_equal(a, b)


def fit_score_on_examples(p: Program, pairs: Sequence[Tuple[Grid, Grid]]) -> Tuple[int, float]:
    """回傳 (成功數, 代價)；代價以長度為主，少量鼓勵形狀匹配與色彩分佈接近。
    用於 Beam 排序：成功數優先，其次代價愈小愈好。
    """
    ok = 0
    penalty = 0.0
    for inp, ref in pairs:
        try:
            out = p.run(inp)
        except Exception:
            penalty += 10.0
            continue
        if grid_equal(out, ref):
            ok += 1
        else:
            # 形狀差異懲罰
            penalty += 0.5 * (abs(out.shape[0] - ref.shape[0]) + abs(out.shape[1] - ref.shape[1]))
            # 顏色分佈（直方圖）差異
            for c in range(10):
                penalty += 0.01 * abs(np.sum(out == c) - np.sum(ref == c))
    # 程式長度懲罰
    penalty += 0.1 * len(p.ops)
    return ok, penalty


# ---------------------------
# Beam Search（帶啟發式）
# ---------------------------

@dataclass
class SearchConfig:
    max_depth: int = MAX_DEPTH
    beam_size: int = BEAM_SIZE
    candidates_per_step: int = CANDIDATES_PER_STEP
    timeout_s: float = TIMEOUT_PER_TASK


def infer_grid_shape(train_pairs: Sequence[Tuple[Grid, Grid]], test_inputs: Sequence[Grid]) -> Optional[Tuple[int,int]]:
    """若訓練輸出形狀一致，嘗試假設測試也相同，供某些原語（置中）使用。"""
    shapes = {ref.shape for (_, ref) in train_pairs}
    if len(shapes) == 1:
        return list(shapes)[0]
    # 或若輸入形狀一致且 ref 與 inp 同形
    in_shapes = {inp.shape for (inp, _ref) in train_pairs}
    if len(in_shapes) == 1 and all(inp.shape == ref.shape for inp, ref in train_pairs):
        return list(in_shapes)[0]
    return None


def search_programs(train_pairs: Sequence[Tuple[Grid, Grid]], cfg: SearchConfig) -> List[Program]:
    start_time = time.time()
    # 初始候選：單步原語
    beam: List[Program] = [Program(())]
    best: List[Tuple[Tuple[int, float], Program]] = []  # ((ok,penalty), program)

    # 預計試圖早停：若已找到完美覆蓋的多個方案，可直接返回前 N
    perfect_needed = NUM_ATTEMPTS

    for depth in range(1, cfg.max_depth + 1):
        # 擴展
        cand: List[Program] = []
        for p in beam:
            for q in program_candidates(p):
                cand.append(q)
        # 隨機下採樣（控制候選體量）
        if len(cand) > cfg.candidates_per_step:
            cand = RNG.sample(cand, cfg.candidates_per_step)

        scored: List[Tuple[Tuple[int,float], Program]] = []
        for q in cand:
            score = fit_score_on_examples(q, train_pairs)
            scored.append((score, q))
        # 排序：ok 由大到小；penalty 由小到大
        scored.sort(key=lambda x: (-x[0][0], x[0][1]))

        # 更新最佳集合
        for s, q in scored[: cfg.beam_size]:
            best.append((s, q))
        best.sort(key=lambda x: (-x[0][0], x[0][1]))
        best = best[: 256]

        # 更新 beam
        beam = [q for (_s, q) in scored[: cfg.beam_size]]

        # 早停條件：
        if best and best[0][0][0] == len(train_pairs):
            # 蒐集前幾個完美方案
            perfects = [q for (s, q) in best if s[0] == len(train_pairs)]
            if len(perfects) >= perfect_needed:
                return perfects[:perfect_needed]

        if time.time() - start_time > cfg.timeout_s:
            break

    # 若無完美方案，回傳排行榜前幾名
    return [q for (_s, q) in best[:NUM_ATTEMPTS]] or [Program(("identity",))]


# ---------------------------
# 推論與提交組裝
# ---------------------------

def solve_task(task: Dict[str, Any], cfg: SearchConfig) -> List[Grid]:
    """針對單一 task，輸出最多 NUM_ATTEMPTS 個答案（pass@2）。"""
    # 解析 train / test
    train_pairs: List[Tuple[Grid,Grid]] = []
    for ex in task["train"]:
        train_pairs.append((list_to_grid(ex["input"]), list_to_grid(ex["output"])) )
    test_inputs: List[Grid] = [list_to_grid(t["input"]) for t in task["test"]]

    # 搜尋程式
    progs = search_programs(train_pairs, cfg)

    # 產出答案（針對全 test，逐一應用相同程式）
    answers: List[Grid] = []
    for p in progs[:NUM_ATTEMPTS]:
        outs: List[Grid] = []
        ok = True
        for inp in test_inputs:
            try:
                out = p.run(inp)
            except Exception:
                ok = False
                break
            outs.append(out)
        if ok:
            # 若多個 test，官方會逐一比對；我們需回傳單一或多個影像？
            # Kaggle ARC 一般每個 test 只有一個輸入；多輸入時需各自對應。
            # 這裡回傳最後一個輸出（或拼接策略）——為保守起見，若多輸入則回傳第一個。
            answers.append(outs[-1] if outs else list_to_grid([[0]]))
    # 備援策略：若不足 NUM_ATTEMPTS，補上 identity 與 fill_majority
    if len(answers) < NUM_ATTEMPTS and test_inputs:
        answers.append(identity(test_inputs[-1]))
    if len(answers) < NUM_ATTEMPTS and test_inputs:
        answers.append(fill_majority(test_inputs[-1]))

    return answers[:NUM_ATTEMPTS]


def to_submission_mapping(answers: Dict[str, List[Grid]]) -> Dict[str, List[List[List[int]]]]:
    out: Dict[str, List[List[List[int]]]] = {}
    for tid, arr in answers.items():
        out[tid] = [grid_to_list(g) for g in arr]
    return out


# ---------------------------
# 主流程與 CLI
# ---------------------------

def discover_eval_split(data_root: str) -> str:
    """盡可能自動偵測可用的評測分割資料夾名稱。常見："public-eval" 或 "evaluation"。
    若找不到，預設使用 "evaluation"。"""
    candidates = [
        "public_evaluation", "public-evaluation", "public_eval", "public-eval",
        "evaluation", "eval"
    ]
    for name in candidates:
        if os.path.isdir(os.path.join(data_root, name)):
            return name
    return "evaluation"


def solve_split(data_root: str, split_name: str, cfg: SearchConfig) -> Dict[str, List[Grid]]:
    tasks = load_arc_split(data_root, split_name)
    answers: Dict[str, List[Grid]] = {}
    t0 = time.time()
    for i, (tid, task) in enumerate(tasks.items(), 1):
        st = time.time()
        try:
            ans = solve_task(task, cfg)
        except Exception as e:
            print(f"[WARN] task {tid} 失敗：{e}", file=sys.stderr)
            # 失敗時以最簡策略填補
            test_inputs = [list_to_grid(t["input"]) for t in task.get("test", [])]
            ans = [identity(test_inputs[-1])] if test_inputs else [np.zeros((1,1), dtype=np.int64)]
        answers[tid] = ans
        dt = time.time() - st
        print(f"[{i}/{len(tasks)}] {tid} 用時 {dt:.2f}s | 程式（示例）: {ans[0].shape if ans else 'N/A'}")
    print(f"總用時：{time.time()-t0:.1f}s；共 {len(tasks)} 題。")
    return answers


def save_submission(mapping: Dict[str, List[List[List[int]]]], out_path: str) -> None:
    with open(out_path, 'w') as f:
        json.dump(mapping, f)
    print(f"已輸出：{out_path}")


def main(argv: Optional[List[str]] = None) -> None:
    parser = argparse.ArgumentParser(description="ARC Prize 2025 基線解題器")
    parser.add_argument("--data-root", type=str, required=True, help="資料根目錄（官方 ARC‑AGI‑2 結構）")
    parser.add_argument("--split", type=str, default=None, help="要解的分割（預設自動偵測）")
    parser.add_argument("--out", type=str, default="submission.json", help="輸出檔名")
    parser.add_argument("--beam", type=int, default=BEAM_SIZE)
    parser.add_argument("--depth", type=int, default=MAX_DEPTH)
    parser.add_argument("--cands", type=int, default=CANDIDATES_PER_STEP)
    parser.add_argument("--timeout", type=float, default=TIMEOUT_PER_TASK)
    args = parser.parse_args(argv)

    cfg = SearchConfig(
        max_depth=args.depth,
        beam_size=args.beam,
        candidates_per_step=args.cands,
        timeout_s=args.timeout,
    )

    split = args.split or discover_eval_split(args.data_root)
    answers = solve_split(args.data_root, split, cfg)
    mapping = to_submission_mapping(answers)
    save_submission(mapping, args.out)


# ---------------------------
# 輕量單元測試（本地/Notebook 可執行，以利公開審閱）
# ---------------------------

def _toy_task_identity() -> Dict[str, Any]:
    inp = [[1,0],[0,1]]
    return {
        "train": [{"input": inp, "output": inp}],
        "test":  [{"input": inp}],
    }


def _toy_task_rotate() -> Dict[str, Any]:
    a = [[1,2],[3,4]]
    b = [[2,4],[1,3]]  # rot90(a)
    return {
        "train": [{"input": a, "output": b}],
        "test":  [{"input": a}],
    }


def _run_toys():
    cfg = SearchConfig(max_depth=3, beam_size=16, candidates_per_step=64, timeout_s=2.0)
    for name, tk in {
        "identity": _toy_task_identity(),
        "rot90": _toy_task_rotate(),
    }.items():
        ans = solve_task(tk, cfg)
        print(name, "->", [grid_to_list(x) for x in ans])


if __name__ == "__main__":
    # 如需快速本地測試，取消下一行註解：
    # _run_toys(); sys.exit(0)
    main()

